# RQ3 — Seleção de PRs para Análise Qualitativa

Este notebook seleciona os *pull-requests* a serem lidos qualitativamente na RQ3, por **amostragem estratificada segundo valores já medidos** das métricas da RQ2, e monta as **URLs do GitHub** de cada caso.

**Nota metodológica importante.** Os estratos abaixo são definidos por **critérios objetivos** (valores numéricos das métricas, que são fatos já calculados na RQ2) — *não* por classificação temática. O rótulo interpretativo de cada estrato é apenas uma **hipótese a verificar na leitura**, não uma categoria atribuída a priori. A classificação do fenômeno (incluindo a categoria de Khelifi) ocorre **somente na Fase 2**, ao ler efetivamente cada PR, podendo confirmar ou refutar a hipótese.

| Estrato (critério objetivo) | Hipótese a verificar na leitura |
|---|---|
| `ia_waste=0` ∧ `human_rework=0` ∧ `commits_humano=0` | autonomia plena ("Piloto") |
| `ia_waste` ≥ P99 | desperdício elevado |
| `human_rework` ≥ P90 (entre rework>0) | retrabalho humano substantivo |
| `agent=Claude_Code` ∧ `commits_humano≥3` | colaboração mais próxima (Achado 3) |
| `commits_humano≥1` ∧ `human_rework` ≤ P25 | guidance invisível à métrica de código |

**Link dos PRs:** como nenhuma fonte traz o número do PR, o link é montado a partir do `merge_sha` (presente em 100% do corpus) — a página do commit de merge no GitHub leva ao PR. Detalhes na Seção 1.

## 0. Setup, download e carga dos dados

Roda no Google Colab. Baixa o `autonomy_waste_rework.csv` (métricas da RQ2) diretamente do host.

In [1]:
import pandas as pd
import numpy as np

# --- Download dos dados (RQ2) ---
!wget -q http://ygor.ml/uff/eo2/results/rq2/autonomy_waste_rework.csv -O autonomy_waste_rework.csv
print("Download concluído.")

# --- Carga ---
df = pd.read_csv("autonomy_waste_rework.csv")
print(f"Linhas no CSV de métricas: {len(df):,}")
print(f"Colunas: {list(df.columns)}")

Download concluído.
Linhas no CSV de métricas: 45,209
Colunas: ['pr_id', 'repo_id', 'repo_full_name', 'agent', 'merge_sha', 'ancestor_sha', 'main_before_sha', 'pr_head_sha', 'is_squash_merge', 'has_internal_merge', 'internal_merge_sha', 'smain_actions', 'smerge_actions', 'spr_ia_actions', 'spr_humano_actions', 'spr_bot_actions', 'human_rework', 'ia_waste', 'ia_accepted_actions', 'overlap_ia_humano', 'number_commits', 'number_commits_ia', 'number_commits_humano', 'number_commits_bot', 'error_message']


In [2]:
# --- Corpus analisável da RQ2 (mesmos filtros do paper) ---
# merge real (sem squash/ff), sem merge interno, sem erro de cálculo
corpus = df[
    (df.is_squash_merge == 0) &
    (df.has_internal_merge == 0) &
    (df.error_message.isna())
].copy()
print(f"Corpus analisável: {len(corpus):,} PRs")
print(corpus.agent.value_counts())

Corpus analisável: 29,923 PRs
agent
Copilot        16822
Devin          11703
Claude_Code     1398
Name: count, dtype: int64


## 1. Montar a URL do PR (via `merge_sha`)

Nenhuma das fontes disponíveis traz o **número** do PR diretamente. Mas o `merge_sha` (presente em 100% do corpus) permite montar um link estável: a página do **commit de merge** no GitHub mostra o PR que o originou.

- **Link primário:** `github.com/{repo}/commit/{merge_sha}` — abre o commit de merge; o GitHub exibe "This commit was created from pull request #N" com link direto ao PR.
- **Link de busca (fallback):** `github.com/{repo}/pulls?q=is:pr+{merge_sha}` — localiza o PR pelo SHA.

Essa abordagem dispensa o `pull_request.parquet` e funciona para todos os casos.

In [3]:
# Corpus já filtrado. Montar as duas URLs a partir do merge_sha.
corpus["url_commit"] = (
    "https://github.com/" + corpus.repo_full_name + "/commit/" + corpus.merge_sha
)
corpus["url_busca_pr"] = (
    "https://github.com/" + corpus.repo_full_name + "/pulls?q=is:pr+" + corpus.merge_sha
)

# URL principal a usar nas fichas (commit de merge -> leva ao PR)
corpus["url"] = corpus["url_commit"]

n_ok = corpus["merge_sha"].notna().sum()
print(f"PRs com merge_sha (e portanto URL): {n_ok:,} de {len(corpus):,}")
corpus[["repo_full_name","agent","merge_sha","url"]].head()

PRs com merge_sha (e portanto URL): 29,923 de 29,923


,repo_full_name,agent,merge_sha,url
3,0x2me/ai-agent-smith,Devin,e4df9e42c7976fec32481a3e6532eb6f2b296d79,https://github.com/0x2me/ai-agent-smith/commit...
4,0GiS0/mcp-sampling,Copilot,8c0dd03ac6dddde649e61319fcaf6bad3c7701e1,https://github.com/0GiS0/mcp-sampling/commit/8...
5,0GiS0/mcp-sampling,Copilot,2ff96a684634146f87193d85fbdadf503543da89,https://github.com/0GiS0/mcp-sampling/commit/2...
6,0xc3u/Indiko.OpenWeatherClient,Copilot,41cc5a2af0d15b101fc0cb0e4d10e719e3677d47,https://github.com/0xc3u/Indiko.OpenWeatherCli...
8,0x3st/nage,Copilot,a1650af816502d6bf284d7e905fdfd01c76abad6,https://github.com/0x3st/nage/commit/a1650af81...


## 2. Definição dos limiares estatísticosOs cortes são definidos pela distribuição das métricas (percentis), pois o objetivo é ler os **extremos**. A quantidade lida por perfil é pequena e parecida entre perfis (amostragem por diversidade de fenômeno, não por representatividade).

In [4]:
# Limiares (percentis) sobre o corpus analisável
P99_waste = corpus.ia_waste.quantile(0.99)

com_rework = corpus[corpus.human_rework > 0]
P90_rework = com_rework.human_rework.quantile(0.90)

print("=== Limiares estatísticos ===")
print(f"Desperdício P99 (ia_waste)        : {P99_waste:,.0f} ações")
print(f"Retrabalho P90 (human_rework>0)   : {P90_rework:,.0f} ações")
print(f"  (base de PRs com rework>0       : {len(com_rework):,})")

# Quantos casos ler por perfil (parecido entre perfis)
N_POR_PERFIL = 3

=== Limiares estatísticos ===
Desperdício P99 (ia_waste)        : 1,895 ações
Retrabalho P90 (human_rework>0)   : 988 ações
  (base de PRs com rework>0       : 2,480)


## 3. Seleção dos 4 perfisCada perfil é ordenado por uma chave relevante e amostrado. Use `random_state` para reprodutibilidade. Onde faz sentido, ordena-se por magnitude (ex.: pegar os mais extremos); onde a diversidade importa, faz-se amostra aleatória dentro do estrato.

In [5]:
SEED = 42

# ---------- Estrato 1: waste=0, rework=0, commits_humano=0 ----------
# Critério OBJETIVO: PRs sem desperdício, sem retrabalho, sem commits humanos.
# Hipótese a verificar na leitura: autonomia plena ("Piloto").
# Filtra ia_accepted_actions moderado (evita PRs triviais) e varia ferramenta.
p1 = corpus[
    (corpus.human_rework == 0) &
    (corpus.ia_waste == 0) &
    (corpus.number_commits_humano == 0) &
    (corpus.ia_accepted_actions >= corpus.ia_accepted_actions.median())  # não-trivial
].copy()

# Amostra 1 caso por ferramenta (diversidade), de forma segura
def amostra_por_ferramenta(d, n_por_agente=1, seed=SEED):
    partes = []
    for ag in d.agent.unique():
        g = d[d.agent == ag]
        partes.append(g.sample(min(len(g), n_por_agente), random_state=seed))
    return pd.concat(partes) if partes else d.head(0)

p1_sel = amostra_por_ferramenta(p1, 1).head(N_POR_PERFIL).copy()
p1_sel["perfil"] = "1_waste0_rework0_humano0"
print(f"Estrato 1 (waste0/rework0/humano0): {len(p1)} candidatos -> {len(p1_sel)} selecionados")
print(f"  Ferramentas: {list(p1_sel.agent)}")

Estrato 1 (waste0/rework0/humano0): 8491 candidatos -> 3 selecionados
  Ferramentas: ['Devin', 'Copilot', 'Claude_Code']


In [6]:
# ---------- Estrato 2: ia_waste >= P99 ----------
# Critério OBJETIVO: desperdício no topo da distribuição.
# Hipótese a verificar na leitura: desperdício elevado (processo vs. demo).
# Evitar SÓ os outliers de milhões (projetos-demo, já explicados):
# pegar casos na faixa "alta mas plausível" e 1 no topo extremo.
p2_alto = corpus[corpus.ia_waste >= P99_waste].copy()

# Faixa plausível: entre P99 e o P99.9 (desperdício real de processo)
P999_waste = corpus.ia_waste.quantile(0.999)
p2_plausivel = p2_alto[p2_alto.ia_waste < P999_waste]
p2_extremo   = p2_alto[p2_alto.ia_waste >= P999_waste]

p2_sel = pd.concat([
    p2_plausivel.sample(min(len(p2_plausivel), 2), random_state=SEED),
    p2_extremo.sample(min(len(p2_extremo), 1), random_state=SEED),
]).head(N_POR_PERFIL)
p2_sel["perfil"] = "2_waste_p99"
print(f"Estrato 2 (ia_waste>=P99): {len(p2_alto)} candidatos -> {len(p2_sel)} selecionados")
print(f"  faixa plausível [{P99_waste:.0f}, {P999_waste:.0f}); extremo >= {P999_waste:.0f}")

Estrato 2 (ia_waste>=P99): 300 candidatos -> 3 selecionados
  faixa plausível [1895, 408179); extremo >= 408179


In [7]:
# ---------- Estrato 3: human_rework >= P90 (entre rework>0) ----------
# Critério OBJETIVO: retrabalho humano no topo da distribuição.
# Hipótese a verificar na leitura: retrabalho substantivo; classificar
# pela taxonomia de Khelifi NA LEITURA. Cruzar com overlap.
p3 = com_rework[com_rework.human_rework >= P90_rework].copy()

# Diversificar por overlap: 1 com overlap alto (correção por cima),
# 1 com overlap baixo (adição nova), 1 livre. Robusto a estratos vazios.
def sample_seguro(d, n, seed):
    if len(d) == 0:
        return d.head(0)
    return d.sample(min(len(d), n), random_state=seed)

if "overlap_ia_humano" in p3.columns and len(p3):
    med_ov = p3.overlap_ia_humano.median()
    p3_alto_ov  = p3[p3.overlap_ia_humano >= med_ov]
    p3_baixo_ov = p3[p3.overlap_ia_humano <  med_ov]
    p3_sel = pd.concat([
        sample_seguro(p3_alto_ov, 1, SEED),
        sample_seguro(p3_baixo_ov, 1, SEED),
        sample_seguro(p3, 1, SEED + 1),
    ]).drop_duplicates("pr_id").head(N_POR_PERFIL).copy()
else:
    p3_sel = sample_seguro(p3, N_POR_PERFIL, SEED).copy()
p3_sel["perfil"] = "3_rework_p90"
print(f"Estrato 3 (human_rework>=P90): {len(p3)} candidatos -> {len(p3_sel)} selecionados")

Estrato 3 (human_rework>=P90): 248 candidatos -> 2 selecionados


In [8]:
# ---------- Estrato 4: agent=Claude_Code & commits_humano>=3 ----------
# Critério OBJETIVO: PRs do Claude Code com >=3 commits humanos.
# Hipótese a verificar na leitura: colaboração mais próxima (Achado 3).
# Cautela: n menor (1.398 PRs do CC); tratar como hipótese.
p4 = corpus[
    (corpus.agent == "Claude_Code") &
    (corpus.number_commits_humano >= 3)
].copy()
# Ordenar por mais commits humanos (casos mais colaborativos)
p4_sel = p4.sort_values("number_commits_humano", ascending=False).head(N_POR_PERFIL).copy()
p4_sel["perfil"] = "4_claudecode_commitshum3+"
print(f"Estrato 4 (ClaudeCode & commits_hum>=3): {len(p4)} candidatos -> {len(p4_sel)} selecionados")

Estrato 4 (ClaudeCode & commits_hum>=3): 283 candidatos -> 3 selecionados


## 4. Busca dirigida: estrato `commits_humano≥1` ∧ `human_rework` baixo (Fase 2.5)

Critério **objetivo**: PRs com commits humanos na branch mas `human_rework` no quartil inferior — ou seja, houve envolvimento humano via commits, mas pouca ação humana sobreviveu ao merge.

**Hipótese a verificar na leitura:** esses casos podem revelar *guidance* (orientação nos comentários/reviews) que a métrica estrutural não captura, pois ela só enxerga código que sobrevive ao merge. A confirmação é **indireta aqui** e só se resolve na leitura — é preciso abrir o PR e verificar se houve discussão extensa. A hipótese pode ser refutada (ex.: o humano apenas fez um commit trivial).

In [9]:
# Estrato 5: commits humanos presentes, mas human_rework baixo/nulo
# (humano participou via commits, mas pouco código sobreviveu)
# Hipótese: possível guidance invisível à métrica -> verificar na leitura.
p5 = corpus[
    (corpus.number_commits_humano >= 1) &
    (corpus.human_rework <= corpus.human_rework.quantile(0.25))
].copy()
# Priorizar os que têm mais commits humanos (mais sinal de envolvimento)
p5_sel = p5.sort_values("number_commits_humano", ascending=False).head(2).copy()
p5_sel["perfil"] = "5_commitshum_rework_baixo"
print(f"Estrato 5 (commits_hum>=1 & rework<=P25): {len(p5)} candidatos -> {len(p5_sel)} selecionados")
print("  (verificar na leitura se há discussão/guidance nos comentários)")

Estrato 5 (commits_hum>=1 & rework<=P25): 166 candidatos -> 2 selecionados
  (verificar na leitura se há discussão/guidance nos comentários)


## 5. Consolidação e exportaçãoJunta todos os perfis, monta a tabela final com as colunas relevantes e as URLs, e exporta para CSV. Esta é a planilha que você usará para abrir cada PR e preencher as fichas de leitura.

In [10]:
# Consolidar todos os perfis
COLS = [
    "perfil", "repo_full_name", "agent", "pr_id",
    "ia_waste", "human_rework", "ia_accepted_actions",
    "spr_ia_actions", "spr_humano_actions",
    "number_commits_humano", "overlap_ia_humano",
    "merge_sha", "url", "url_busca_pr",
]
COLS = [c for c in COLS if c in corpus.columns or c in {"perfil","url","url_busca_pr"}]

selecionados = pd.concat([p1_sel, p2_sel, p3_sel, p4_sel, p5_sel], ignore_index=True)
selecionados = selecionados[[c for c in COLS if c in selecionados.columns]]

print(f"Total de PRs selecionados para leitura: {len(selecionados)}")
selecionados

Total de PRs selecionados para leitura: 13


,perfil,repo_full_name,agent,pr_id,ia_waste,human_rework,ia_accepted_actions,spr_ia_actions,spr_humano_actions,number_commits_humano,overlap_ia_humano,merge_sha,url,url_busca_pr
0,1_waste0_rework0_humano0,opathways/web-app,Devin,3177249509,0,0,223,223,0,0,0,17989986cbbde22acd39748ca66aecbca032a854,https://github.com/opathways/web-app/commit/17...,https://github.com/opathways/web-app/pulls?q=i...
1,1_waste0_rework0_humano0,Bmn599/Fernly,Copilot,3184078414,0,0,134,134,0,0,0,0b3b2818c0b38a0f6da019d9f037ce70769bb0c5,https://github.com/Bmn599/Fernly/commit/0b3b28...,https://github.com/Bmn599/Fernly/pulls?q=is:pr...
2,1_waste0_rework0_humano0,UltimatePea/chinese-ocaml,Claude_Code,3238800383,0,0,827,827,0,0,0,3a76909117a21509300abd1018be5781d0cc1d14,https://github.com/UltimatePea/chinese-ocaml/c...,https://github.com/UltimatePea/chinese-ocaml/p...
3,2_waste_p99,Inglan/ama-webapp,Copilot,3218774844,2996,29,309,3305,44,3,8,7966b8198cfa0e1b38ec695fc6386cfc13f5234b,https://github.com/Inglan/ama-webapp/commit/79...,https://github.com/Inglan/ama-webapp/pulls?q=i...
4,2_waste_p99,davides93/expense-tracker-workshop,Copilot,3118108053,1902,0,250,2152,0,0,0,9b02e67d679794121c83d9cc9b5417c893749edc,https://github.com/davides93/expense-tracker-w...,https://github.com/davides93/expense-tracker-w...
5,2_waste_p99,r3e-network/r3e-faas,Devin,2886621484,2207457,0,6241,2213698,0,0,0,294ab0cb20ce33ba125afa8c4407902263adc849,https://github.com/r3e-network/r3e-faas/commit...,https://github.com/r3e-network/r3e-faas/pulls?...
6,3_rework_p90,Jibran1998/nuvianix-tech,Devin,3002188243,0,3920,0,0,3972,3,0,cfe811bb564f60e766adc775d6273333f15c728d,https://github.com/Jibran1998/nuvianix-tech/co...,https://github.com/Jibran1998/nuvianix-tech/pu...
7,3_rework_p90,pasoko/nw_simulator,Claude_Code,3170058878,1346,4619,933,2279,505458,18,74,78b4070dfab0e208c4f5179017726697366dd198,https://github.com/pasoko/nw_simulator/commit/...,https://github.com/pasoko/nw_simulator/pulls?q...
8,4_claudecode_commitshum3+,WilliamAGH/williamcallahan.com,Claude_Code,3181761564,0,5391,0,0,5879,47,0,c9f03d95f41a20a34a52f3eb248e2e8b01113097,https://github.com/WilliamAGH/williamcallahan....,https://github.com/WilliamAGH/williamcallahan....
9,4_claudecode_commitshum3+,WilliamAGH/williamcallahan.com,Claude_Code,3207026702,0,2560,0,0,3035,45,0,fbb27c8f1984965c703baf58472620b67bdbf5a5,https://github.com/WilliamAGH/williamcallahan....,https://github.com/WilliamAGH/williamcallahan....


In [11]:
# Conferência: todos têm merge_sha (e portanto URL)?
sem_url = selecionados[selecionados.url.isna()]
if len(sem_url):
    print(f"⚠️  {len(sem_url)} PRs sem URL — improvável, verifique merge_sha:")
    print(sem_url[["perfil","repo_full_name","pr_id"]].to_string(index=False))
else:
    print("✓ Todos os PRs selecionados têm URL (via merge_sha).")

# Exportar
selecionados.to_csv("rq3_casos_selecionados.csv", index=False)
print("\nExportado: rq3_casos_selecionados.csv")

✓ Todos os PRs selecionados têm URL (via merge_sha).

Exportado: rq3_casos_selecionados.csv


In [12]:
# ============================================================
# Gerar XLSX pré-preenchido para as fichas de leitura (Fase 2)
# Colunas de DADOS (do CSV) + colunas de FICHA (vazias, p/ preencher na leitura)
# ============================================================
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

wb = Workbook()
ws = wb.active
ws.title = "Casos RQ3"

col_dados = ["perfil", "repo_full_name", "agent", "pr_id", "ia_waste", "human_rework",
             "ia_accepted_actions", "spr_ia_actions", "spr_humano_actions",
             "number_commits_humano", "overlap_ia_humano", "merge_sha", "url", "url_busca_pr"]
col_ficha = ["Contexto (tarefa/repo)", "Texto (discussão/revisão)",
             "Categoria Khelifi", "Interpretação", "Hipótese confirmada?"]
headers = col_dados + col_ficha

FONT = "Arial"
fill_dados = PatternFill("solid", fgColor="1F4E78")   # azul: dados do CSV
fill_ficha = PatternFill("solid", fgColor="C55A11")   # laranja: campos a preencher
white_bold = Font(name=FONT, bold=True, color="FFFFFF", size=10)
thin = Side(style="thin", color="D9D9D9")
border = Border(left=thin, right=thin, top=thin, bottom=thin)

# Cabeçalho
for j, h in enumerate(headers, start=1):
    c = ws.cell(row=1, column=j, value=h)
    c.font = white_bold
    c.fill = fill_dados if h in col_dados else fill_ficha
    c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    c.border = border

# Linhas de dados (col_dados pré-preenchidas; col_ficha em branco)
for i, (_, row) in enumerate(selecionados.iterrows(), start=2):
    for j, col in enumerate(col_dados, start=1):
        val = row.get(col, "")
        if pd.isna(val): val = ""
        c = ws.cell(row=i, column=j, value=val)
        c.font = Font(name=FONT, size=10)
        c.alignment = Alignment(vertical="top",
                                wrap_text=(col in {"url", "url_busca_pr", "repo_full_name"}))
        c.border = border
    for j in range(len(col_dados) + 1, len(headers) + 1):
        c = ws.cell(row=i, column=j, value="")
        c.font = Font(name=FONT, size=10)
        c.alignment = Alignment(vertical="top", wrap_text=True)
        c.border = border

# Larguras de coluna
larguras = {"perfil": 26, "repo_full_name": 28, "agent": 12, "pr_id": 13, "ia_waste": 11,
            "human_rework": 13, "ia_accepted_actions": 16, "spr_ia_actions": 14,
            "spr_humano_actions": 16, "number_commits_humano": 18, "overlap_ia_humano": 15,
            "merge_sha": 20, "url": 42, "url_busca_pr": 42}
for j, h in enumerate(headers, start=1):
    ws.column_dimensions[get_column_letter(j)].width = larguras.get(h, 26)

ws.freeze_panes = "B2"   # congela cabeçalho e 1a coluna
wb.save("rq3_casos_selecionados.xlsx")
print(f"Exportado: rq3_casos_selecionados.xlsx ({len(selecionados)} casos)")
print(f"  {len(col_dados)} colunas de dados pré-preenchidas (azul)")
print(f"  {len(col_ficha)} colunas de ficha para preencher na leitura (laranja)")

Exportado: rq3_casos_selecionados.xlsx (13 casos)
  14 colunas de dados pré-preenchidas (azul)
  5 colunas de ficha para preencher na leitura (laranja)


In [13]:
# Imprimir as URLs agrupadas por perfil (para abrir no navegador)
for perfil, grupo in selecionados.groupby("perfil"):
    print(f"\n{'='*64}\n{perfil}\n{'='*64}")
    for _, row in grupo.iterrows():
        print(f"  {row['agent']:12} | waste={row.get('ia_waste',0):>9.0f} "
              f"rework={row.get('human_rework',0):>6.0f} "
              f"commits_hum={row.get('number_commits_humano',0):>3.0f}")
        print(f"    merge : {row['url']}")
        if 'url_busca_pr' in row and pd.notna(row.get('url_busca_pr')):
            print(f"    busca : {row['url_busca_pr']}")


1_waste0_rework0_humano0
  Devin        | waste=        0 rework=     0 commits_hum=  0
    merge : https://github.com/opathways/web-app/commit/17989986cbbde22acd39748ca66aecbca032a854
    busca : https://github.com/opathways/web-app/pulls?q=is:pr+17989986cbbde22acd39748ca66aecbca032a854
  Copilot      | waste=        0 rework=     0 commits_hum=  0
    merge : https://github.com/Bmn599/Fernly/commit/0b3b2818c0b38a0f6da019d9f037ce70769bb0c5
    busca : https://github.com/Bmn599/Fernly/pulls?q=is:pr+0b3b2818c0b38a0f6da019d9f037ce70769bb0c5
  Claude_Code  | waste=        0 rework=     0 commits_hum=  0
    merge : https://github.com/UltimatePea/chinese-ocaml/commit/3a76909117a21509300abd1018be5781d0cc1d14
    busca : https://github.com/UltimatePea/chinese-ocaml/pulls?q=is:pr+3a76909117a21509300abd1018be5781d0cc1d14

2_waste_p99
  Copilot      | waste=     2996 rework=    29 commits_hum=  3
    merge : https://github.com/Inglan/ama-webapp/commit/7966b8198cfa0e1b38ec695fc6386cfc13f5234b
 

## 6. Ficha de leitura — usar o XLSX gerado

A célula anterior gerou **`rq3_casos_selecionados.xlsx`**, já pré-preenchido. Abra-o e preencha as colunas em laranja (uma linha por PR):

- **Colunas azuis (prontas):** perfil, repo, agente, métricas do PR, `merge_sha` e as URLs.
- **Colunas laranja (preencher na leitura):** Contexto, Texto (discussão/revisão), Categoria Khelifi, Interpretação, Hipótese confirmada?

**Como ler cada PR:** abra a **URL do commit de merge** — a página no GitHub mostra "This commit was created from pull request #N", levando ao PR. Se não aparecer, use a **URL de busca**.

**Disciplina metodológica (atende à crítica sobre classificar antes de ler):**
- O estrato em que o PR caiu é um **fato numérico**, não uma classificação temática.
- A coluna *Categoria Khelifi* e a *Interpretação* só são preenchidas **após** ler o PR.
- A coluna *Hipótese confirmada?* registra explicitamente se a leitura **confirmou ou refutou** o rótulo esperado do estrato. Casos refutados são achados legítimos e interessantes (ex.: um PR de "waste=0/rework=0" que, na leitura, revela guidance intenso nos comentários).

**Lembrete de escopo:** análise ilustrativa, não taxonômica. Não reportar frequências de categoria — apenas interpretar os casos.